# 01 Padding 类型和用法：为什么卷积前要在边缘补东西

Padding 中文通常叫“填充”。

在卷积神经网络里，它指的是：

```text
在输入图片或特征图的边缘，额外补上一圈或多圈数值。
```

这节只讲 padding 本身，不展开完整 CNN 网络。

先记一句话：

```text
Padding 的核心作用，是控制卷积后的尺寸，并让边缘信息不至于太快丢失。
```

## 1. 为什么会需要 padding

卷积核在图片上滑动时，需要覆盖一块完整区域。

比如一个 3 x 3 卷积核，要看 3 x 3 的小窗口。

如果图片大小是 5 x 5，不加 padding 时，卷积核不能把中心放到最边缘的位置。

因为一旦中心太靠边，3 x 3 窗口就会伸到图片外面。

所以不加 padding 时，边缘像素被使用的次数更少，输出尺寸也会变小。

这就是 padding 出现的第一个原因：

```text
让卷积核也能更充分地看见边缘区域。
```

## 2. 不加 padding 会发生什么

假设输入图片大小是：

$$
5\times5
$$

卷积核大小是：

$$
3\times3
$$

步长是：

$$
S=1
$$

不加 padding 时，输出大小会变成：

$$
3\times3
$$

也就是说，高和宽都少了 2。

如果网络有很多层卷积，每一层都让高宽变小，特征图会很快缩水。

这会带来两个问题：

```text
空间信息越来越少
边缘信息越来越容易被忽略
```

## 3. 输出尺寸公式

先看一维方向，比如只看高度。

输入高度是 H，卷积核大小是 K，padding 是 P，stride 是 S。

输出高度为：

$$
H_{out}=\left\lfloor\frac{H+2P-K}{S}\right\rfloor+1
$$

宽度方向同理：

$$
W_{out}=\left\lfloor\frac{W+2P-K}{S}\right\rfloor+1
$$

这个公式里最关键的是：

$$
H+2P
$$

为什么是 2P？

因为高度方向上方补 P 行，下方也补 P 行。

宽度方向也是一样，左边补 P 列，右边补 P 列。

## 4. 第一种：Valid Padding

Valid padding 其实就是不加 padding。

也就是：

$$
P=0
$$

卷积核只在完全覆盖输入区域的位置上计算。

所以输出尺寸通常会变小。

例如：

```text
输入：28 x 28
卷积核：3 x 3
stride：1
padding：0
输出：26 x 26
```

Valid 的特点是：

```text
不补边，输出变小，只保留卷积核能完整覆盖的位置。
```

## 5. Valid Padding 什么时候用

Valid padding 适合你愿意让特征图自然变小的时候。

比如有时我们希望卷积层一边提取特征，一边稍微压缩空间尺寸。

它的优点是简单，不引入额外边界值。

它的缺点是边缘信息更容易丢失，堆多层后尺寸缩水很快。

一句话记：

```text
Valid = 不补边，输出会变小。
```

## 6. 第二种：Same Padding

Same padding 的目标是：

```text
让输出高宽尽量和输入高宽保持一样。
```

最常见情况是 stride 等于 1，卷积核大小是奇数。

比如 3 x 3 卷积核，只要四周补一圈：

$$
P=1
$$

就可以保持尺寸不变。

例如：

```text
输入：28 x 28
卷积核：3 x 3
stride：1
padding：1
输出：28 x 28
```

如果是 5 x 5 卷积核，通常补两圈：

$$
P=2
$$

这就是 Same padding 最常见的用法。

## 7. Same Padding 的常用计算

当 stride 等于 1，卷积核大小是奇数时，如果想保持输入输出大小一样，可以用：

$$
P=\frac{K-1}{2}
$$

例如 3 x 3 卷积核：

$$
P=\frac{3-1}{2}=1
$$

例如 5 x 5 卷积核：

$$
P=\frac{5-1}{2}=2
$$

这就是为什么很多 CNN 里常见：

```text
3 x 3 卷积核配 padding = 1
5 x 5 卷积核配 padding = 2
```

它们都是为了在 stride = 1 时保持高宽不变。

## 8. Same Padding 什么时候用

Same padding 很常用。

它适合这些情况：

```text
希望多堆几层卷积，但不想高宽很快变小
希望边缘像素也有更多机会参与计算
希望网络结构里某些层的输入输出尺寸容易对齐
```

在很多经典 CNN 中，3 x 3 卷积核配 padding = 1 是非常常见的组合。

一句话记：

```text
Same = 补边，让输出尺寸尽量保持不变。
```

## 9. 第三种：Full Padding

Full padding 的目标不是保持尺寸，而是让卷积核能扫到更多边缘外扩位置。

可以理解成：

```text
补得更多，让卷积核只要和原输入有一点重叠就参与计算。
```

如果卷积核大小是 K，Full padding 常见补法是：

$$
P=K-1
$$

比如 3 x 3 卷积核：

$$
P=2
$$

这会让输出尺寸比输入更大。

Full padding 在普通 CNN 分类网络里没有 Same 那么常见，但理解它有助于区分 padding 的不同目标。

## 10. Full Padding 什么时候用

Full padding 更常见于一些信号处理、数学卷积解释，或者需要让输出尺寸扩大的场景。

在图像分类入门阶段，你不需要把它当作最常用技巧。

先记住它和 Valid、Same 的区别即可：

```text
Valid：不补，输出变小
Same：适当补，输出尽量不变
Full：补得更多，输出变大
```

## 11. 第四种：Zero Padding

Zero padding 指的是用 0 来填充边缘。

这是最常见的 padding 内容。

比如原来是一张 3 x 3 的小图：

```text
1 2 3
4 5 6
7 8 9
```

如果四周补一圈 0，会变成：

```text
0 0 0 0 0
0 1 2 3 0
0 4 5 6 0
0 7 8 9 0
0 0 0 0 0
```

Zero padding 简单、稳定、常用。

CNN 里说 padding = 1，很多时候默认就是四周补一圈 0。

## 12. Zero Padding 的优缺点

Zero padding 的优点是简单。

0 通常表示没有额外信号，所以它不会直接制造强烈特征。

但它也有一个问题：

```text
边缘突然出现一圈 0，可能让边界处的数值分布和图像内部不一样。
```

对大多数普通 CNN 来说，这个问题通常可以接受。

但在一些图像生成、图像复原、超分辨率等任务里，边缘效果更重要，就可能考虑其他 padding 方式。

## 13. 第五种：Reflection Padding

Reflection padding 叫反射填充。

它不是补 0，而是把边缘附近的值像镜子一样反射出去。

直觉是：

```text
边缘外面缺什么，就从边缘里面镜像补一点。
```

它的好处是边缘过渡更自然，不会突然出现一圈 0。

Reflection padding 常见于图像生成、风格迁移、图像复原等更在意边缘视觉效果的任务。

一句话记：

```text
Reflection = 像镜子一样补边，边缘更自然。
```

## 14. 第六种：Replication Padding

Replication padding 叫复制填充，也可以理解成边缘复制。

它会把最边缘的值复制到外面。

直觉是：

```text
图片边缘是什么值，外面就继续沿用这个值。
```

它比 Zero padding 更保留边界处的数值连续性。

但它可能让边缘区域被重复强调。

一句话记：

```text
Replication = 复制最外层边缘值。
```

## 15. 第七种：Circular Padding

Circular padding 叫循环填充。

它会把一边的内容拿到另一边补。

比如左边缺值，就从右边拿；上边缺值，就从下边拿。

直觉是：

```text
把图像或信号看成首尾相接的环。
```

它在普通图像分类中不常用。

但在周期性信号、环形数据、某些特殊物理场景里可能有意义。

一句话记：

```text
Circular = 首尾相接地补。
```

## 16. 第八种：Causal Padding

Causal padding 常见于一维序列模型，比如时间序列、语音、文本卷积。

它的目标是：

```text
当前位置的输出不能看到未来信息。
```

比如预测今天时，只能看今天和过去，不能提前看到明天。

所以 causal padding 通常只在序列前面补，不在后面补。

图像 CNN 入门阶段不需要重点掌握它。

但如果以后学时序模型，要记得它是为了避免信息泄露。

## 17. 对称 padding 和非对称 padding

很多时候我们默认上下左右补一样多。

这叫对称 padding。

例如：

```text
上 1，下 1，左 1，右 1
```

但有时为了精确控制输出尺寸，也会使用非对称 padding。

例如：

```text
上 0，下 1，左 0，右 1
```

非对称 padding 常见于 stride 不等于 1、卷积核大小为偶数、或者需要严格对齐尺寸的情况。

初学阶段先掌握对称 padding，理解非对称 padding 是为了更精细地控制尺寸即可。

## 18. Padding 的常见用途

Padding 常见用途可以总结成四类。

第一，控制输出尺寸。

比如 3 x 3 卷积核配 padding = 1，让 stride = 1 时高宽保持不变。

第二，保留边缘信息。

不加 padding 时，边缘像素参与卷积的机会少。

第三，方便堆叠多层网络。

如果每层卷积都让尺寸变小，网络还没堆几层，特征图就缩水很多。

第四，方便不同分支的特征图对齐。

后面学更复杂网络时，经常会遇到不同分支结果要相加或拼接，这时候尺寸对齐很重要。

## 19. 怎么选择 padding

初学阶段可以按这个顺序判断。

如果你只是做普通图像分类，并且用 3 x 3 卷积核：

```text
优先想到 padding = 1
```

如果你希望输出尺寸自然变小：

```text
用 valid，也就是 padding = 0
```

如果你希望输出尺寸保持不变：

```text
用 same 思路，根据卷积核大小选择 padding
```

如果任务特别在意图像边缘效果：

```text
可以考虑 reflection 或 replication
```

如果是时间序列，不能偷看未来：

```text
考虑 causal padding
```

## 20. 常见类型对比

| 类型 | 怎么补 | 输出尺寸特点 | 常见用途 |
| --- | --- | --- | --- |
| Valid | 不补 | 变小 | 自然压缩尺寸 |
| Same | 补到尺寸尽量不变 | 基本不变 | 普通 CNN 常用 |
| Full | 补得更多 | 变大 | 信号处理或特殊场景 |
| Zero | 用 0 补 | 取决于补多少 | 最常见填充值 |
| Reflection | 镜像补 | 取决于补多少 | 图像生成、复原 |
| Replication | 复制边缘值 | 取决于补多少 | 边缘连续性更重要时 |
| Circular | 首尾相接 | 取决于补多少 | 周期性数据 |
| Causal | 只补过去方向 | 保证不看未来 | 时间序列、语音、文本 |

注意：Valid、Same、Full 更像是在说“补多少”。

Zero、Reflection、Replication、Circular 更像是在说“补什么”。

这两个角度不要混在一起。

## 21. 本节总结

Padding 的逻辑链是：

```text
卷积核在边缘处不容易完整覆盖输入
-> 不加 padding 时输出尺寸会变小
-> 边缘信息也更容易被忽略
-> padding 通过补边控制输出尺寸和边缘参与程度
-> valid 表示不补
-> same 表示尽量保持尺寸
-> full 表示补得更多让输出变大
-> zero / reflection / replication / circular 说明补什么值
-> causal padding 用在不能看未来的序列任务中
```

先记住一句话：

```text
Padding 不是为了凭空增加信息，而是为了控制卷积核在边缘和尺寸上的行为。
```

## 22. 自查问题

1. 为什么不加 padding 时，卷积输出尺寸会变小？
2. 28 x 28 输入，3 x 3 卷积核，stride = 1，padding = 0，输出尺寸是多少？
3. 28 x 28 输入，3 x 3 卷积核，stride = 1，padding = 1，输出尺寸是多少？
4. Valid、Same、Full 分别是在说什么？
5. Zero padding 和 Reflection padding 的区别是什么？
6. 为什么普通 3 x 3 卷积常配 padding = 1？
7. Causal padding 为什么不能补未来方向？